# Topic Trend Monitoring

## Business question

Which complaint topics are accelerating over time and may require early operational intervention?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

scores_path = Path("reports/sample/complaint_scores.csv")
terms_path = Path("reports/sample/topic_terms.csv")
if not scores_path.exists() or not terms_path.exists():
    raise FileNotFoundError(
        "Run `cx-complaints analyze --output-dir reports/sample` first."
    )

scores = pd.read_csv(scores_path)
topic_terms = pd.read_csv(terms_path)
scores["date_received"] = pd.to_datetime(scores["date_received"], errors="coerce")
scores = scores.dropna(subset=["date_received", "topic_id"]).copy()
scores["topic_id"] = scores["topic_id"].astype(int)
scores["month"] = scores["date_received"].dt.to_period("M").astype(str)

plt.style.use("ggplot")

## 1. Build topic labels

In [ ]:
topic_labels = (
    topic_terms.sort_values(["topic_id", "rank"])
    .groupby("topic_id")["term"]
    .apply(lambda s: ", ".join(s.head(5).astype(str)))
    .rename("topic_label")
    .reset_index()
)
topic_labels.head()

## 2. Monthly topic share table

In [ ]:
monthly_topic_counts = (
    scores.groupby(["month", "topic_id"])
    .size()
    .rename("complaints")
    .reset_index()
)

monthly_totals = monthly_topic_counts.groupby("month")["complaints"].sum().rename("month_total")
monthly_topic = monthly_topic_counts.merge(monthly_totals, on="month", how="left")
monthly_topic["topic_share_pct"] = (
    monthly_topic["complaints"] / monthly_topic["month_total"] * 100
).round(2)
monthly_topic = monthly_topic.merge(topic_labels, on="topic_id", how="left")
monthly_topic.head(20)

## 3. Trend deltas and emerging-topic flags

In [ ]:
monthly_topic = monthly_topic.sort_values(["topic_id", "month"]).copy()
monthly_topic["share_delta_pp"] = (
    monthly_topic.groupby("topic_id")["topic_share_pct"].diff().fillna(0)
)

trend_summary = (
    monthly_topic.groupby(["topic_id", "topic_label"], dropna=False)
    .agg(
        avg_share_pct=("topic_share_pct", "mean"),
        latest_share_pct=("topic_share_pct", "last"),
        previous_share_pct=("topic_share_pct", lambda s: s.iloc[-2] if len(s) > 1 else s.iloc[-1]),
        latest_delta_pp=("share_delta_pp", "last"),
        observations=("month", "nunique"),
    )
    .reset_index()
    .sort_values("latest_delta_pp", ascending=False)
)

trend_summary["emerging_flag"] = (
    (trend_summary["latest_delta_pp"] >= 2.0)
    & (trend_summary["latest_share_pct"] >= 5.0)
)
trend_summary.head(15)

In [ ]:
emerging = trend_summary[trend_summary["emerging_flag"]].copy()
emerging

## 4. Visualize top topic shares over time

In [ ]:
top_topics = (
    monthly_topic.groupby("topic_id")["complaints"].sum().sort_values(ascending=False).head(5).index.tolist()
)
plot_data = monthly_topic[monthly_topic["topic_id"].isin(top_topics)].copy()

fig, ax = plt.subplots(figsize=(12, 6))
for topic_id, group in plot_data.groupby("topic_id"):
    label = group["topic_label"].iloc[0]
    short_label = f"Topic {topic_id}: {label[:45]}"
    ax.plot(group["month"], group["topic_share_pct"], marker="o", label=short_label)

ax.set_title("Top Topic Share Trends")
ax.set_xlabel("Month")
ax.set_ylabel("Topic share (%)")
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()

## 5. Suggested actions

For topics with `emerging_flag = True`:

1. Review representative complaints and affected product/company mix.
2. Validate whether changes are sustained for 2+ months.
3. Trigger operational owners for root-cause checks and response scripts.

In [ ]:
# Optional export
output_dir = Path("reports/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

monthly_topic.to_csv(output_dir / "notebook_monthly_topic_share.csv", index=False)
trend_summary.to_csv(output_dir / "notebook_topic_trend_summary.csv", index=False)

output_dir